In [10]:
import pandas as pd
import re
import os

In [11]:
print(os.listdir(".."))

['.git', '.gitattributes', '.gitignore', 'allsides_8values.csv', 'app.py', 'BABE_combined.csv', 'Backend', 'data', 'notebook', 'notebooks', 'README.md', 'requirements.txt', 'SBIC_full_dataset.csv', 'wino_bias_full.csv']


In [12]:
babe_df = pd.read_csv("../BABE_combined.csv")
allsides_df = pd.read_csv("../allsides_8values.csv")
sbic_df = pd.read_csv("../SBIC_full_dataset.csv")
wino_df = pd.read_csv("../wino_bias_full.csv")

In [13]:
print("BABE Columns:")
print(babe_df.columns)

print("\nAllSides Columns:")
print(allsides_df.columns)

print("\nSBIC Columns:")
print(sbic_df.columns)

print("\nWinoBias Columns:")
print(wino_df.columns)

BABE Columns:
Index(['text', 'label', 'topic', 'biased_words', 'uuid', 'type',
       'label_opinion'],
      dtype='object')

AllSides Columns:
Index(['Idx', 'Event', 'Root_Topic', 'Sub_Topic', 'Date', 'Split_Tag'], dtype='object')

SBIC Columns:
Index(['Unnamed: 0', 'post', 'targetMinority', 'targetCategory',
       'targetStereotype', 'whoTarget', 'intentYN', 'sexYN', 'offensiveYN',
       'dataSource', 'hasBiasedImplication'],
      dtype='object')

WinoBias Columns:
Index(['document_id', 'part_number', 'word_number', 'tokens', 'pos_tags',
       'parse_bit', 'predicate_lemma', 'predicate_framenet_id', 'word_sense',
       'speaker', 'ner_tags', 'verbal_predicates', 'coreference_clusters'],
      dtype='object')


In [14]:
babe_df = babe_df[['text', 'label']]

babe_df.head()

,text,label
0,"As the Black Lives Matter movement grows, comp...",0
1,The case of Rahaf Mohammed al-Qunun drawn new ...,0
2,The Post said the talks on payroll taxes were ...,0
3,Nearly 78 percent of Americans report experien...,0
4,Colin P. Clarke has been teaching a course on ...,0


In [15]:
sbic_df = sbic_df[['post', 'hasBiasedImplication']]

sbic_df.columns = ['text', 'label']

sbic_df.head()

,text,label
0,"\n\nBill Kristol and Ben Shaprio, two turds in...",1
1,\n\nRose\n🌹Taylor‏ @RealRoseTaylor 6h6 hours a...,1
2,\nCharlie Kirk‏\n\nJohnny Depp calls for death...,0
3,\nDavid Knight‏ \n\nNotice how quickly things ...,1
4,\nFinland fireball: Time-lapse video shows nig...,1


In [16]:
wino_df = wino_df[['tokens']]

wino_df['text'] = wino_df['tokens']

wino_df['label'] = 1

wino_df = wino_df[['text', 'label']]

wino_df.head()

,text,label
0,['The' 'janitor' 'reprimanded' 'the' 'accounta...,1
1,['The' 'carpenter' 'always' 'ask' 'the' 'libra...,1
2,['The' 'carpenter' 'always' 'asks' 'the' 'libr...,1
3,['The' 'physician' 'wanted' 'to' 'meet' 'the' ...,1
4,['The' 'physician' 'wanted' 'to' 'meet' 'the' ...,1


In [17]:
allsides_df.head()

,Idx,Event,Root_Topic,Sub_Topic,Date,Split_Tag
0,0,Google Holds Illegal Monopoly Over Online Ad T...,Social,Technology,2025/4/17,train_data
1,1,Alleged 4chan Hack Exposes Emails of Moderators,Social,Technology,2025/4/16,test_data
2,2,Chinese Electric Vehicle Company Surpasses Tes...,Social,Technology,2025/3/25,test_data
3,3,Mark Rober’s Tesla Drove Itself Through a Wall...,Social,Technology,2025/3/19,train_data
4,4,Hinge and Tinder to Roll Out AI ‘Wingmen’ to H...,Social,Technology,2025/3/13,train_data


In [19]:
print(allsides_df.columns)

Index(['Idx', 'Event', 'Root_Topic', 'Sub_Topic', 'Date', 'Split_Tag'], dtype='object')


In [20]:
allsides_df['text'] = (
    allsides_df['Root_Topic'].astype(str) + " news about " +
    allsides_df['Event'].astype(str)
)

allsides_df['label'] = 1

allsides_df = allsides_df[['text', 'label']]

allsides_df.head()

,text,label
0,Social news about Google Holds Illegal Monopol...,1
1,Social news about Alleged 4chan Hack Exposes E...,1
2,Social news about Chinese Electric Vehicle Com...,1
3,Social news about Mark Rober’s Tesla Drove Its...,1
4,Social news about Hinge and Tinder to Roll Out...,1


In [21]:
df = pd.concat(
    [babe_df, sbic_df, wino_df, allsides_df],
    ignore_index=True
)

df.head()

,text,label
0,"As the Black Lives Matter movement grows, comp...",0
1,The case of Rahaf Mohammed al-Qunun drawn new ...,0
2,The Post said the talks on payroll taxes were ...,0
3,Nearly 78 percent of Americans report experien...,0
4,Colin P. Clarke has been teaching a course on ...,0


In [22]:
df.shape

(51456, 2)

In [23]:
df['label'].value_counts()

label
1    33778
0    17678
Name: count, dtype: int64

In [24]:
df.dropna(inplace=True)

In [25]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['text'] = df['text'].apply(clean_text)

df.head()

,text,label
0,as the black lives matter movement grows compa...,0
1,the case of rahaf mohammed alqunun drawn new g...,0
2,the post said the talks on payroll taxes were ...,0
3,nearly percent of americans report experienci...,0
4,colin p clarke has been teaching a course on t...,0


In [26]:
df.drop_duplicates(inplace=True)

print("Duplicate rows:", df.duplicated().sum())

print("Final unique rows:", len(df))

Duplicate rows: 0
Final unique rows: 51156


In [27]:
biased = df[df['label'] == 1]
neutral = df[df['label'] == 0]

min_size = min(len(biased), len(neutral))

biased = biased.sample(min_size, random_state=42)
neutral = neutral.sample(min_size, random_state=42)

df = pd.concat([biased, neutral])

df['label'].value_counts()

label
1    17543
0    17543
Name: count, dtype: int64

In [28]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

train.shape, test.shape

((28069, 2), (7017, 2))

In [29]:
df.to_csv("../data/data.csv", index=False)